In [1]:
# ============================================================
# FOOD PRICE FORECASTING
# 04 - FEATURE ENGINEERING
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "agmarknet_cleaned_observations.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "food_price_forecasting_features.csv"

print("=" * 90)
print("FOOD PRICE FORECASTING - FEATURE ENGINEERING")
print("=" * 90)

print(f"\nInput file:\n{INPUT_FILE}")
print(f"Output file:\n{OUTPUT_FILE}")


# ============================================================
# 2. LOAD DEFINITIVE CLEANED DATASET
# ============================================================

df = pd.read_csv(INPUT_FILE)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print("\n" + "=" * 90)
print("DATASET LOADED")
print("=" * 90)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Crops: {df['Crop'].nunique()}")
print(f"States: {df['State'].nunique()}")
print(f"Markets: {df['Market'].nunique():,}")


# ============================================================
# 3. BASIC VALIDATION
# ============================================================

required_columns = [
    "Date",
    "Crop",
    "State",
    "Market",
    "Variety",
    "Grade",
    "Minimum Price",
    "Maximum Price",
    "Modal Price",
    "Price_Range_Anomaly",
]

missing_required = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required:
    raise ValueError(
        f"Required columns are missing: {missing_required}"
    )

df = df[df["Modal Price"] > 0].copy()

df = df.sort_values(
    [
        "Crop",
        "State",
        "Market",
        "Variety",
        "Grade",
        "Date",
    ]
).reset_index(drop=True)

print("\nValidation complete.")
print(f"Usable observations: {len(df):,}")


# ============================================================
# 4. IDENTIFY EXTREME PRICE OBSERVATIONS
# ============================================================
# We DO NOT remove extreme prices.
# They are retained and represented through explicit features.

df["Is_Extreme_Price"] = (
    df["Modal Price"] >= 10000
).astype("int8")

df["Is_Very_Extreme_Price"] = (
    df["Modal Price"] >= 50000
).astype("int8")

df["Log_Modal_Price"] = np.log1p(
    df["Modal Price"]
)

print("\n" + "=" * 90)
print("EXTREME PRICE FLAGS")
print("=" * 90)

print(
    f"Extreme observations (≥ ₹10,000): "
    f"{df['Is_Extreme_Price'].sum():,}"
)

print(
    f"Very extreme observations (≥ ₹50,000): "
    f"{df['Is_Very_Extreme_Price'].sum():,}"
)


# ============================================================
# 5. CALENDAR FEATURES
# ============================================================

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Quarter"] = df["Date"].dt.quarter
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["DayOfYear"] = df["Date"].dt.dayofyear
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

# Month beginning
df["Month_Start"] = df["Date"].dt.to_period("M").dt.to_timestamp()

# Continuous time index
df["Time_Index"] = (
    (df["Year"] - df["Year"].min()) * 12
    + df["Month"]
)

# Season
season_map = {
    1: "Winter",
    2: "Winter",
    3: "Spring",
    4: "Spring",
    5: "Summer",
    6: "Summer",
    7: "Monsoon",
    8: "Monsoon",
    9: "Monsoon",
    10: "Autumn",
    11: "Autumn",
    12: "Winter",
}

df["Season"] = df["Month"].map(season_map)

print("\nCalendar features created.")


# ============================================================
# 6. MONTHLY CROP-LEVEL PRICE DATA
# ============================================================
# The final forecasting model will work at a monthly crop level.
#
# Daily/market observations are first aggregated using MEDIAN.
# Median is deliberately used because the EDA showed a long
# right tail and several extreme observations.

monthly = (
    df.groupby(
        ["Crop", "Month_Start"],
        as_index=False
    )
    .agg(
        Median_Price=("Modal Price", "median"),
        Mean_Price=("Modal Price", "mean"),
        Min_Price=("Modal Price", "min"),
        Max_Price=("Modal Price", "max"),
        Price_Observations=("Modal Price", "count"),
        Markets=("Market", "nunique"),
        States=("State", "nunique"),
        Varieties=("Variety", "nunique"),
        Extreme_Observations=("Is_Extreme_Price", "sum"),
        Very_Extreme_Observations=("Is_Very_Extreme_Price", "sum"),
        Anomaly_Observations=("Price_Range_Anomaly", "sum"),
    )
)

monthly = monthly.sort_values(
    ["Crop", "Month_Start"]
).reset_index(drop=True)

monthly["Year"] = monthly["Month_Start"].dt.year
monthly["Month"] = monthly["Month_Start"].dt.month
monthly["Quarter"] = monthly["Month_Start"].dt.quarter

monthly["Season"] = monthly["Month"].map(season_map)

print("\n" + "=" * 90)
print("MONTHLY DATASET CREATED")
print("=" * 90)

print(f"Rows: {len(monthly):,}")
print(f"Crops: {monthly['Crop'].nunique()}")
print(
    f"Date range: "
    f"{monthly['Month_Start'].min().date()} → "
    f"{monthly['Month_Start'].max().date()}"
)


# ============================================================
# 7. CHECK MONTHLY COVERAGE
# ============================================================

coverage = (
    monthly.groupby("Crop")
    .agg(
        First_Month=("Month_Start", "min"),
        Last_Month=("Month_Start", "max"),
        Months=("Month_Start", "nunique"),
    )
    .sort_index()
)

print("\n" + "=" * 90)
print("CROP MONTHLY COVERAGE")
print("=" * 90)

print(coverage.to_string())

# Identify crops with a complete 2015-2025 monthly history.
expected_months = 11 * 12
complete_crops = (
    coverage.loc[
        (coverage["Months"] == expected_months)
        & (coverage["First_Month"] == pd.Timestamp("2015-01-01"))
        & (coverage["Last_Month"] == pd.Timestamp("2025-12-01")),
    ]
    .index
    .tolist()
)

incomplete_crops = [
    crop for crop in monthly["Crop"].unique()
    if crop not in complete_crops
]

print(
    f"\nComplete 2015-2025 crops: {len(complete_crops)}"
)
print(complete_crops)

print(
    f"\nIncomplete crops retained for audit/documentation only: "
    f"{len(incomplete_crops)}"
)
print(incomplete_crops)

# Do not impute missing calendar months. In particular, Lentil has
# an incomplete historical series and must not be allowed to turn a
# multi-year gap into a false "next month" target through shift(-1).
# The complete crop set is used for the supervised forecasting dataset.
forecast_monthly = monthly[
    monthly["Crop"].isin(complete_crops)
].copy()

forecast_monthly = forecast_monthly.sort_values(
    ["Crop", "Month_Start"]
).reset_index(drop=True)

print(
    f"\nMonthly rows retained for forecasting: "
    f"{len(forecast_monthly):,}"
)


# ============================================================
# 8. LAG FEATURES
# ============================================================
# All lag features use ONLY historical values.
# No future information is allowed to enter a row.

group = forecast_monthly.groupby("Crop", group_keys=False)

for lag in [1, 2, 3, 6, 12]:
    forecast_monthly[f"Price_Lag_{lag}M"] = (
        group["Median_Price"].shift(lag)
    )

print("\nLag features created:")
print("1M, 2M, 3M, 6M and 12M")


# ============================================================
# 9. PRICE CHANGE FEATURES
# ============================================================

forecast_monthly["Price_Change_1M"] = (
    forecast_monthly["Median_Price"]
    / forecast_monthly["Price_Lag_1M"]
    - 1
)

forecast_monthly["Price_Change_3M"] = (
    forecast_monthly["Median_Price"]
    / forecast_monthly["Price_Lag_3M"]
    - 1
)

forecast_monthly["Price_Change_12M"] = (
    forecast_monthly["Median_Price"]
    / forecast_monthly["Price_Lag_12M"]
    - 1
)

forecast_monthly["Price_Difference_1M"] = (
    forecast_monthly["Median_Price"]
    - forecast_monthly["Price_Lag_1M"]
)


# ============================================================
# 10. ROLLING PRICE FEATURES
# ============================================================
# IMPORTANT:
# shift(1) ensures the current month's target is NOT included
# in the rolling historical statistics.

historical_price = group["Median_Price"].shift(1)

forecast_monthly["Rolling_Mean_3M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(3, min_periods=2)
    .mean()
    .reset_index(level=0, drop=True)
)

forecast_monthly["Rolling_Mean_6M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(6, min_periods=3)
    .mean()
    .reset_index(level=0, drop=True)
)

forecast_monthly["Rolling_Mean_12M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(12, min_periods=6)
    .mean()
    .reset_index(level=0, drop=True)
)

forecast_monthly["Rolling_Median_3M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(3, min_periods=2)
    .median()
    .reset_index(level=0, drop=True)
)

forecast_monthly["Rolling_Median_6M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(6, min_periods=3)
    .median()
    .reset_index(level=0, drop=True)
)

forecast_monthly["Rolling_Median_12M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(12, min_periods=6)
    .median()
    .reset_index(level=0, drop=True)
)


# ============================================================
# 11. ROLLING VOLATILITY
# ============================================================

forecast_monthly["Rolling_Std_3M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(3, min_periods=2)
    .std()
    .reset_index(level=0, drop=True)
)

forecast_monthly["Rolling_Std_6M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(6, min_periods=3)
    .std()
    .reset_index(level=0, drop=True)
)

forecast_monthly["Rolling_Std_12M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(12, min_periods=6)
    .std()
    .reset_index(level=0, drop=True)
)


# ============================================================
# 12. ROLLING MIN / MAX
# ============================================================

forecast_monthly["Rolling_Min_6M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(6, min_periods=3)
    .min()
    .reset_index(level=0, drop=True)
)

forecast_monthly["Rolling_Max_6M"] = (
    historical_price
    .groupby(forecast_monthly["Crop"])
    .rolling(6, min_periods=3)
    .max()
    .reset_index(level=0, drop=True)
)


# ============================================================
# 13. PRICE MOMENTUM FEATURES
# ============================================================

forecast_monthly["Momentum_3M"] = (
    forecast_monthly["Price_Lag_1M"]
    - forecast_monthly["Price_Lag_3M"]
)

forecast_monthly["Momentum_6M"] = (
    forecast_monthly["Price_Lag_1M"]
    - forecast_monthly["Price_Lag_6M"]
)

forecast_monthly["Momentum_12M"] = (
    forecast_monthly["Price_Lag_1M"]
    - forecast_monthly["Price_Lag_12M"]
)


# ============================================================
# 14. YEAR-OVER-YEAR SEASONAL FEATURES
# ============================================================

forecast_monthly["Same_Month_Last_Year"] = (
    forecast_monthly["Price_Lag_12M"]
)

forecast_monthly["YoY_Price_Change"] = (
    forecast_monthly["Median_Price"]
    / forecast_monthly["Same_Month_Last_Year"]
    - 1
)


# ============================================================
# 15. HISTORICAL CROP PRICE STATISTICS
# ============================================================
# These are expanding statistics shifted by one period, meaning
# the current month's target is never used to calculate them.

forecast_monthly["Historical_Mean_Price"] = (
    group["Median_Price"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=3).mean()
    )
)

forecast_monthly["Historical_Median_Price"] = (
    group["Median_Price"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=3).median()
    )
)

forecast_monthly["Historical_Std_Price"] = (
    group["Median_Price"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=3).std()
    )
)


# ============================================================
# 16. MONTHLY SEASONAL PRICE BASELINE
# ============================================================
# Historical same-calendar-month information only.
#
# Example:
# For March 2024, this feature uses March observations from
# previous years, not March 2024 itself.

forecast_monthly["Historical_Month_Mean"] = (
    monthly
    .groupby(["Crop", "Month"])["Median_Price"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=2).mean()
    )
)

forecast_monthly["Historical_Month_Median"] = (
    monthly
    .groupby(["Crop", "Month"])["Median_Price"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=2).median()
    )
)


# ============================================================
# 17. PRICE DEVIATION FEATURES
# ============================================================

forecast_monthly["Deviation_From_Rolling_Mean_6M"] = (
    forecast_monthly["Price_Lag_1M"]
    - forecast_monthly["Rolling_Mean_6M"]
)

forecast_monthly["Deviation_From_Rolling_Median_6M"] = (
    forecast_monthly["Price_Lag_1M"]
    - forecast_monthly["Rolling_Median_6M"]
)

forecast_monthly["Rolling_CV_6M"] = (
    forecast_monthly["Rolling_Std_6M"]
    / forecast_monthly["Rolling_Mean_6M"]
)


# ============================================================
# 18. MARKET ACTIVITY FEATURES
# ============================================================
# These describe the amount of information available for each
# crop-month. They are useful explanatory variables.

forecast_monthly["Observation_Intensity"] = np.log1p(
    forecast_monthly["Price_Observations"]
)

forecast_monthly["Market_Count_Log"] = np.log1p(
    forecast_monthly["Markets"]
)

forecast_monthly["State_Count_Log"] = np.log1p(
    forecast_monthly["States"]
)

forecast_monthly["Variety_Count_Log"] = np.log1p(
    forecast_monthly["Varieties"]
)

forecast_monthly["Extreme_Rate"] = (
    forecast_monthly["Extreme_Observations"]
    / forecast_monthly["Price_Observations"]
)

forecast_monthly["Anomaly_Rate"] = (
    forecast_monthly["Anomaly_Observations"]
    / forecast_monthly["Price_Observations"]
)


# ============================================================
# 19. TIME TREND FEATURES
# ============================================================

forecast_monthly["Time_Index"] = (
    (forecast_monthly["Year"] - forecast_monthly["Year"].min()) * 12
    + forecast_monthly["Month"]
)

forecast_monthly["Time_Index_Squared"] = (
    forecast_monthly["Time_Index"] ** 2
)


# ============================================================
# 20. CYCLICAL MONTH ENCODING
# ============================================================
# Allows models to understand that December and January are
# adjacent months.

forecast_monthly["Month_Sin"] = np.sin(
    2 * np.pi * forecast_monthly["Month"] / 12
)

forecast_monthly["Month_Cos"] = np.cos(
    2 * np.pi * forecast_monthly["Month"] / 12
)

forecast_monthly["Quarter_Sin"] = np.sin(
    2 * np.pi * forecast_monthly["Quarter"] / 4
)

forecast_monthly["Quarter_Cos"] = np.cos(
    2 * np.pi * forecast_monthly["Quarter"] / 4
)


# ============================================================
# 21. FORECAST TARGET
# ============================================================
# Before creating the target, verify that each complete-series row
# is followed by the immediately next calendar month. This prevents
# shift(-1) from turning a missing month into a false next-month target.

next_observed_month = (
    forecast_monthly.groupby("Crop")["Month_Start"].shift(-1)
)

expected_next_month = (
    forecast_monthly["Month_Start"]
    + pd.DateOffset(months=1)
)

continuity_errors = (
    next_observed_month.notna()
    & (next_observed_month != expected_next_month)
).sum()

print(
    f"\nCalendar continuity breaks before target creation: "
    f"{continuity_errors:,}"
)

if continuity_errors:
    raise ValueError(
        "Forecasting data contains non-consecutive calendar months. "
        "Target creation was stopped to prevent an invalid next-month target."
    )

# The target is NEXT MONTH'S median crop price.
#
# For a row representing January:
# Target = February price
#
# This is created by shifting the observed price BACKWARD
# relative to the current feature row.

forecast_monthly["Target_Next_Month_Price"] = (
    group["Median_Price"].shift(-1)
)

forecast_monthly["Target_Next_Month_Log_Price"] = np.log1p(
    forecast_monthly["Target_Next_Month_Price"]
)


# ============================================================
# 22. TARGET CHANGE
# ============================================================

forecast_monthly["Target_Next_Month_Change"] = (
    forecast_monthly["Target_Next_Month_Price"]
    / forecast_monthly["Median_Price"]
    - 1
)


# ============================================================
# 23. TARGET DATE
# ============================================================

forecast_monthly["Target_Date"] = (
    forecast_monthly["Month_Start"]
    + pd.DateOffset(months=1)
)


# ============================================================
# 24. REMOVE ROWS WITHOUT A FORECAST TARGET
# ============================================================
# The final observed month for each crop has no known next-month
# price, so it cannot be used as a supervised training example.

before_target_filter = len(monthly)

forecast_df = forecast_monthly[
    forecast_monthly["Target_Next_Month_Price"].notna()
].copy()

forecast_df = forecast_df.reset_index(drop=True)

removed_target_rows = (
    before_target_filter - len(forecast_df)
)

print("\n" + "=" * 90)
print("TARGET CONSTRUCTION")
print("=" * 90)

print(
    f"Rows before target filtering: "
    f"{before_target_filter:,}"
)

print(
    f"Rows removed without next-month target: "
    f"{removed_target_rows:,}"
)

print(
    f"Forecast-ready rows: "
    f"{len(forecast_df):,}"
)


# ============================================================
# 25. FEATURE VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("FEATURE VALIDATION")
print("=" * 90)

feature_columns = [
    "Price_Lag_1M",
    "Price_Lag_2M",
    "Price_Lag_3M",
    "Price_Lag_6M",
    "Price_Lag_12M",
    "Rolling_Mean_3M",
    "Rolling_Mean_6M",
    "Rolling_Mean_12M",
    "Rolling_Median_3M",
    "Rolling_Median_6M",
    "Rolling_Median_12M",
    "Rolling_Std_3M",
    "Rolling_Std_6M",
    "Rolling_Std_12M",
    "Rolling_Min_6M",
    "Rolling_Max_6M",
    "Historical_Mean_Price",
    "Historical_Median_Price",
    "Historical_Std_Price",
    "Historical_Month_Mean",
    "Historical_Month_Median",
]

feature_missing = (
    forecast_df[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nMissing values in historical/time-series features:")
print(feature_missing.to_string())


# ============================================================
# 26. CHECK FOR INFINITE VALUES
# ============================================================

numeric_columns = forecast_df.select_dtypes(
    include=[np.number]
).columns

infinite_counts = np.isinf(
    forecast_df[numeric_columns]
).sum()

infinite_counts = (
    infinite_counts[infinite_counts > 0]
    .sort_values(ascending=False)
)

print("\nInfinite values:")
if len(infinite_counts) == 0:
    print("None")
else:
    print(infinite_counts.to_string())


# ============================================================
# 27. FEATURE LIST
# ============================================================

target_columns = [
    "Target_Next_Month_Price",
    "Target_Next_Month_Log_Price",
    "Target_Next_Month_Change",
]

excluded_columns = [
    "Target_Date",
]

identifier_columns = [
    "Crop",
    "Month_Start",
    "Year",
    "Month",
    "Quarter",
    "Season",
]

model_feature_columns = [
    col
    for col in forecast_df.columns
    if col not in target_columns
    and col not in excluded_columns
    and col not in [
        "Median_Price",
        "Mean_Price",
        "Min_Price",
        "Max_Price",
    ]
]

print("\n" + "=" * 90)
print("FEATURE SUMMARY")
print("=" * 90)

print(
    f"Total columns: {len(forecast_df.columns)}"
)

print(
    f"Potential model features: "
    f"{len(model_feature_columns)}"
)

print("\nModel feature columns:")
for i, col in enumerate(model_feature_columns, 1):
    print(f"{i:02d}. {col}")


# ============================================================
# 28. CROP-WISE FORECASTING DATASET SUMMARY
# ============================================================

crop_feature_summary = (
    forecast_df.groupby("Crop")
    .agg(
        Rows=("Crop", "size"),
        First_Date=("Month_Start", "min"),
        Last_Date=("Month_Start", "max"),
        Target_Available=("Target_Next_Month_Price", "count"),
        Median_Target=("Target_Next_Month_Price", "median"),
        Mean_Target=("Target_Next_Month_Price", "mean"),
    )
    .sort_values("Rows", ascending=False)
)

print("\n" + "=" * 90)
print("FORECASTING DATASET BY CROP")
print("=" * 90)

print(
    crop_feature_summary.round(2).to_string()
)


# ============================================================
# 29. TIME-ORDER VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("TIME-ORDER VALIDATION")
print("=" * 90)

time_errors = 0

for crop, crop_df in forecast_df.groupby("Crop"):
    crop_df = crop_df.sort_values("Month_Start")

    if not crop_df["Month_Start"].is_monotonic_increasing:
        time_errors += 1

print(
    f"Crops with invalid chronological ordering: "
    f"{time_errors}"
)

# Check target date relationship
target_date_errors = (
    forecast_df["Target_Date"]
    != (
        forecast_df["Month_Start"]
        + pd.DateOffset(months=1)
    )
).sum()

print(
    f"Rows with invalid target-date relationship: "
    f"{target_date_errors:,}"
)


# ============================================================
# 30. SAVE FORECASTING DATASET
# ============================================================

forecast_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 90)
print("FORECASTING FEATURE DATASET SAVED")
print("=" * 90)

print(f"Path: {OUTPUT_FILE}")
print(f"Rows: {len(forecast_df):,}")
print(f"Columns: {len(forecast_df.columns)}")
print(
    f"File size: "
    f"{OUTPUT_FILE.stat().st_size / (1024 ** 2):.2f} MB"
)


# ============================================================
# 31. SAVE FEATURE LIST
# ============================================================

feature_list_file = (
    OUTPUT_DIR
    / "forecasting_feature_columns.csv"
)

pd.DataFrame({
    "Feature": model_feature_columns
}).to_csv(
    feature_list_file,
    index=False
)

print(
    f"\nFeature list saved to:\n"
    f"{feature_list_file}"
)


# ============================================================
# 32. FINAL DATASET PREVIEW
# ============================================================

print("\n" + "=" * 90)
print("FINAL FORECASTING DATASET PREVIEW")
print("=" * 90)

preview_columns = [
    "Crop",
    "Month_Start",
    "Median_Price",
    "Price_Lag_1M",
    "Price_Lag_3M",
    "Price_Lag_12M",
    "Rolling_Mean_3M",
    "Rolling_Mean_6M",
    "Rolling_Std_6M",
    "Month_Sin",
    "Month_Cos",
    "Target_Next_Month_Price",
]

print(
    forecast_df[preview_columns]
    .head(15)
    .round(2)
    .to_string(index=False)
)


# ============================================================
# 33. FINAL VALIDATION SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 90)

print(f"Original cleaned observations: {len(df):,}")
print(f"Monthly crop observations: {len(monthly):,}")
print(f"Forecast-ready observations: {len(forecast_df):,}")
print(f"Number of crops: {forecast_df['Crop'].nunique()}")
print(f"Complete forecasting crops: {len(complete_crops)}")
print(f"Number of model features: {len(model_feature_columns)}")
print(f"Target: Next month's median crop price")

print("\nLeakage protections:")
print("✓ Lag features use historical observations only")
print("✓ Rolling features are shifted by one month")
print("✓ Historical expanding statistics exclude current target")
print("✓ Seasonal historical statistics exclude current observation")
print("✓ Target is the following month's price")
print("✓ Final month of each crop is excluded from supervised training")

print("\nData treatment:")
print("✓ Extreme prices retained")
print("✓ Extreme prices explicitly flagged")
print("✓ Log price feature created")
print("✓ Only complete 2015–2025 crop series are used for supervised forecasting")
print("✓ Incomplete crop histories are retained for documentation but excluded from target construction")
print("✓ No missing calendar months were imputed")
print("✓ Original cleaned dataset was not modified")

print("\n" + "=" * 90)
print("READY FOR MODEL DEVELOPMENT")
print("=" * 90)

FOOD PRICE FORECASTING - FEATURE ENGINEERING

Input file:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed\agmarknet_cleaned_observations.csv
Output file:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed\food_price_forecasting_features.csv

DATASET LOADED
Rows: 484,504
Columns: 15
Date range: 2015-01-01 → 2025-12-08
Crops: 10
States: 32
Markets: 3,220

Validation complete.
Usable observations: 484,504

EXTREME PRICE FLAGS
Extreme observations (≥ ₹10,000): 976
Very extreme observations (≥ ₹50,000): 21

Calendar features created.

MONTHLY DATASET CREATED
Rows: 1,210
Crops: 10
Date range: 2015-01-01 → 2025-12-01

CROP MONTHLY COVERAGE
              First_Month Last_Month  Months
Crop                                        
Banana         2015-01-01 2025-12-01     132
Cotton         2015-01-01 2025-12-01     132
Groundnut      2015-01-01 2025-12-01     132
Jowar/Sorghum  2015-01-01 2025-12-01     132
Lentil         2015-

In [ ]:
# ============================================================
# FOOD PRICE FORECASTING
# 04 - FEATURE ENGINEERING
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "agmarknet_cleaned_observations.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "food_price_forecasting_features.csv"

print("=" * 90)
print("FOOD PRICE FORECASTING - FEATURE ENGINEERING")
print("=" * 90)

print(f"\nInput file:\n{INPUT_FILE}")
print(f"Output file:\n{OUTPUT_FILE}")


# ============================================================
# 2. LOAD DEFINITIVE CLEANED DATASET
# ============================================================

df = pd.read_csv(INPUT_FILE)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print("\n" + "=" * 90)
print("DATASET LOADED")
print("=" * 90)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Crops: {df['Crop'].nunique()}")
print(f"States: {df['State'].nunique()}")
print(f"Markets: {df['Market'].nunique():,}")


# ============================================================
# 3. BASIC VALIDATION
# ============================================================

required_columns = [
    "Date",
    "Crop",
    "State",
    "Market",
    "Variety",
    "Grade",
    "Minimum Price",
    "Maximum Price",
    "Modal Price",
    "Price_Range_Anomaly",
]

missing_required = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required:
    raise ValueError(
        f"Required columns are missing: {missing_required}"
    )

df = df[df["Modal Price"] > 0].copy()

df = df.sort_values(
    [
        "Crop",
        "State",
        "Market",
        "Variety",
        "Grade",
        "Date",
    ]
).reset_index(drop=True)

print("\nValidation complete.")
print(f"Usable observations: {len(df):,}")


# ============================================================
# 4. IDENTIFY EXTREME PRICE OBSERVATIONS
# ============================================================
# We DO NOT remove extreme prices.
# They are retained and represented through explicit features.

df["Is_Extreme_Price"] = (
    df["Modal Price"] >= 10000
).astype("int8")

df["Is_Very_Extreme_Price"] = (
    df["Modal Price"] >= 50000
).astype("int8")

df["Log_Modal_Price"] = np.log1p(
    df["Modal Price"]
)

print("\n" + "=" * 90)
print("EXTREME PRICE FLAGS")
print("=" * 90)

print(
    f"Extreme observations (≥ ₹10,000): "
    f"{df['Is_Extreme_Price'].sum():,}"
)

print(
    f"Very extreme observations (≥ ₹50,000): "
    f"{df['Is_Very_Extreme_Price'].sum():,}"
)


# ============================================================
# 5. CALENDAR FEATURES
# ============================================================

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Quarter"] = df["Date"].dt.quarter
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["DayOfYear"] = df["Date"].dt.dayofyear
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

# Month beginning
df["Month_Start"] = df["Date"].dt.to_period("M").dt.to_timestamp()

# Continuous time index
df["Time_Index"] = (
    (df["Year"] - df["Year"].min()) * 12
    + df["Month"]
)

# Season
season_map = {
    1: "Winter",
    2: "Winter",
    3: "Spring",
    4: "Spring",
    5: "Summer",
    6: "Summer",
    7: "Monsoon",
    8: "Monsoon",
    9: "Monsoon",
    10: "Autumn",
    11: "Autumn",
    12: "Winter",
}

df["Season"] = df["Month"].map(season_map)

print("\nCalendar features created.")


# ============================================================
# 6. MONTHLY CROP-LEVEL PRICE DATA
# ============================================================
# The final forecasting model will work at a monthly crop level.
#
# Daily/market observations are first aggregated using MEDIAN.
# Median is deliberately used because the EDA showed a long
# right tail and several extreme observations.

monthly = (
    df.groupby(
        ["Crop", "Month_Start"],
        as_index=False
    )
    .agg(
        Median_Price=("Modal Price", "median"),
        Mean_Price=("Modal Price", "mean"),
        Min_Price=("Modal Price", "min"),
        Max_Price=("Modal Price", "max"),
        Price_Observations=("Modal Price", "count"),
        Markets=("Market", "nunique"),
        States=("State", "nunique"),
        Varieties=("Variety", "nunique"),
        Extreme_Observations=("Is_Extreme_Price", "sum"),
        Very_Extreme_Observations=("Is_Very_Extreme_Price", "sum"),
        Anomaly_Observations=("Price_Range_Anomaly", "sum"),
    )
)

monthly = monthly.sort_values(
    ["Crop", "Month_Start"]
).reset_index(drop=True)

monthly["Year"] = monthly["Month_Start"].dt.year
monthly["Month"] = monthly["Month_Start"].dt.month
monthly["Quarter"] = monthly["Month_Start"].dt.quarter

monthly["Season"] = monthly["Month"].map(season_map)

print("\n" + "=" * 90)
print("MONTHLY DATASET CREATED")
print("=" * 90)

print(f"Rows: {len(monthly):,}")
print(f"Crops: {monthly['Crop'].nunique()}")
print(
    f"Date range: "
    f"{monthly['Month_Start'].min().date()} → "
    f"{monthly['Month_Start'].max().date()}"
)


# ============================================================
# 7. CHECK MONTHLY COVERAGE
# ============================================================

coverage = (
    monthly.groupby("Crop")
    .agg(
        First_Month=("Month_Start", "min"),
        Last_Month=("Month_Start", "max"),
        Months=("Month_Start", "nunique"),
    )
    .sort_index()
)

print("\n" + "=" * 90)
print("CROP MONTHLY COVERAGE")
print("=" * 90)

print(coverage.to_string())

print(
    "\nImportant: Potato naturally ends in February 2020 "
    "in the available AGMARKNET data. No values are imputed "
    "for the missing period."
)


# ============================================================
# 8. LAG FEATURES
# ============================================================
# All lag features use ONLY historical values.
# No future information is allowed to enter a row.

group = monthly.groupby("Crop", group_keys=False)

for lag in [1, 2, 3, 6, 12]:
    monthly[f"Price_Lag_{lag}M"] = (
        group["Median_Price"].shift(lag)
    )

print("\nLag features created:")
print("1M, 2M, 3M, 6M and 12M")


# ============================================================
# 9. PRICE CHANGE FEATURES
# ============================================================

monthly["Price_Change_1M"] = (
    monthly["Median_Price"]
    / monthly["Price_Lag_1M"]
    - 1
)

monthly["Price_Change_3M"] = (
    monthly["Median_Price"]
    / monthly["Price_Lag_3M"]
    - 1
)

monthly["Price_Change_12M"] = (
    monthly["Median_Price"]
    / monthly["Price_Lag_12M"]
    - 1
)

monthly["Price_Difference_1M"] = (
    monthly["Median_Price"]
    - monthly["Price_Lag_1M"]
)


# ============================================================
# 10. ROLLING PRICE FEATURES
# ============================================================
# IMPORTANT:
# shift(1) ensures the current month's target is NOT included
# in the rolling historical statistics.

historical_price = group["Median_Price"].shift(1)

monthly["Rolling_Mean_3M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(3, min_periods=2)
    .mean()
    .reset_index(level=0, drop=True)
)

monthly["Rolling_Mean_6M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(6, min_periods=3)
    .mean()
    .reset_index(level=0, drop=True)
)

monthly["Rolling_Mean_12M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(12, min_periods=6)
    .mean()
    .reset_index(level=0, drop=True)
)

monthly["Rolling_Median_3M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(3, min_periods=2)
    .median()
    .reset_index(level=0, drop=True)
)

monthly["Rolling_Median_6M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(6, min_periods=3)
    .median()
    .reset_index(level=0, drop=True)
)

monthly["Rolling_Median_12M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(12, min_periods=6)
    .median()
    .reset_index(level=0, drop=True)
)


# ============================================================
# 11. ROLLING VOLATILITY
# ============================================================

monthly["Rolling_Std_3M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(3, min_periods=2)
    .std()
    .reset_index(level=0, drop=True)
)

monthly["Rolling_Std_6M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(6, min_periods=3)
    .std()
    .reset_index(level=0, drop=True)
)

monthly["Rolling_Std_12M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(12, min_periods=6)
    .std()
    .reset_index(level=0, drop=True)
)


# ============================================================
# 12. ROLLING MIN / MAX
# ============================================================

monthly["Rolling_Min_6M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(6, min_periods=3)
    .min()
    .reset_index(level=0, drop=True)
)

monthly["Rolling_Max_6M"] = (
    historical_price
    .groupby(monthly["Crop"])
    .rolling(6, min_periods=3)
    .max()
    .reset_index(level=0, drop=True)
)


# ============================================================
# 13. PRICE MOMENTUM FEATURES
# ============================================================

monthly["Momentum_3M"] = (
    monthly["Price_Lag_1M"]
    - monthly["Price_Lag_3M"]
)

monthly["Momentum_6M"] = (
    monthly["Price_Lag_1M"]
    - monthly["Price_Lag_6M"]
)

monthly["Momentum_12M"] = (
    monthly["Price_Lag_1M"]
    - monthly["Price_Lag_12M"]
)


# ============================================================
# 14. YEAR-OVER-YEAR SEASONAL FEATURES
# ============================================================

monthly["Same_Month_Last_Year"] = (
    monthly["Price_Lag_12M"]
)

monthly["YoY_Price_Change"] = (
    monthly["Median_Price"]
    / monthly["Same_Month_Last_Year"]
    - 1
)


# ============================================================
# 15. HISTORICAL CROP PRICE STATISTICS
# ============================================================
# These are expanding statistics shifted by one period, meaning
# the current month's target is never used to calculate them.

monthly["Historical_Mean_Price"] = (
    group["Median_Price"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=3).mean()
    )
)

monthly["Historical_Median_Price"] = (
    group["Median_Price"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=3).median()
    )
)

monthly["Historical_Std_Price"] = (
    group["Median_Price"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=3).std()
    )
)


# ============================================================
# 16. MONTHLY SEASONAL PRICE BASELINE
# ============================================================
# Historical same-calendar-month information only.
#
# Example:
# For March 2024, this feature uses March observations from
# previous years, not March 2024 itself.

monthly["Historical_Month_Mean"] = (
    monthly
    .groupby(["Crop", "Month"])["Median_Price"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=2).mean()
    )
)

monthly["Historical_Month_Median"] = (
    monthly
    .groupby(["Crop", "Month"])["Median_Price"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=2).median()
    )
)


# ============================================================
# 17. PRICE DEVIATION FEATURES
# ============================================================

monthly["Deviation_From_Rolling_Mean_6M"] = (
    monthly["Price_Lag_1M"]
    - monthly["Rolling_Mean_6M"]
)

monthly["Deviation_From_Rolling_Median_6M"] = (
    monthly["Price_Lag_1M"]
    - monthly["Rolling_Median_6M"]
)

monthly["Rolling_CV_6M"] = (
    monthly["Rolling_Std_6M"]
    / monthly["Rolling_Mean_6M"]
)


# ============================================================
# 18. MARKET ACTIVITY FEATURES
# ============================================================
# These describe the amount of information available for each
# crop-month. They are useful explanatory variables.

monthly["Observation_Intensity"] = np.log1p(
    monthly["Price_Observations"]
)

monthly["Market_Count_Log"] = np.log1p(
    monthly["Markets"]
)

monthly["State_Count_Log"] = np.log1p(
    monthly["States"]
)

monthly["Variety_Count_Log"] = np.log1p(
    monthly["Varieties"]
)

monthly["Extreme_Rate"] = (
    monthly["Extreme_Observations"]
    / monthly["Price_Observations"]
)

monthly["Anomaly_Rate"] = (
    monthly["Anomaly_Observations"]
    / monthly["Price_Observations"]
)


# ============================================================
# 19. TIME TREND FEATURES
# ============================================================

monthly["Time_Index"] = (
    (monthly["Year"] - monthly["Year"].min()) * 12
    + monthly["Month"]
)

monthly["Time_Index_Squared"] = (
    monthly["Time_Index"] ** 2
)


# ============================================================
# 20. CYCLICAL MONTH ENCODING
# ============================================================
# Allows models to understand that December and January are
# adjacent months.

monthly["Month_Sin"] = np.sin(
    2 * np.pi * monthly["Month"] / 12
)

monthly["Month_Cos"] = np.cos(
    2 * np.pi * monthly["Month"] / 12
)

monthly["Quarter_Sin"] = np.sin(
    2 * np.pi * monthly["Quarter"] / 4
)

monthly["Quarter_Cos"] = np.cos(
    2 * np.pi * monthly["Quarter"] / 4
)


# ============================================================
# 21. FORECAST TARGET
# ============================================================
# The target is NEXT MONTH'S median crop price.
#
# For a row representing January:
# Target = February price
#
# This is created by shifting the observed price BACKWARD
# relative to the current feature row.

monthly["Target_Next_Month_Price"] = (
    group["Median_Price"].shift(-1)
)

monthly["Target_Next_Month_Log_Price"] = np.log1p(
    monthly["Target_Next_Month_Price"]
)


# ============================================================
# 22. TARGET CHANGE
# ============================================================

monthly["Target_Next_Month_Change"] = (
    monthly["Target_Next_Month_Price"]
    / monthly["Median_Price"]
    - 1
)


# ============================================================
# 23. TARGET DATE
# ============================================================

monthly["Target_Date"] = (
    monthly["Month_Start"]
    + pd.DateOffset(months=1)
)


# ============================================================
# 24. REMOVE ROWS WITHOUT A FORECAST TARGET
# ============================================================
# The final observed month for each crop has no known next-month
# price, so it cannot be used as a supervised training example.

before_target_filter = len(monthly)

forecast_df = monthly[
    monthly["Target_Next_Month_Price"].notna()
].copy()

forecast_df = forecast_df.reset_index(drop=True)

removed_target_rows = (
    before_target_filter - len(forecast_df)
)

print("\n" + "=" * 90)
print("TARGET CONSTRUCTION")
print("=" * 90)

print(
    f"Rows before target filtering: "
    f"{before_target_filter:,}"
)

print(
    f"Rows removed without next-month target: "
    f"{removed_target_rows:,}"
)

print(
    f"Forecast-ready rows: "
    f"{len(forecast_df):,}"
)


# ============================================================
# 25. FEATURE VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("FEATURE VALIDATION")
print("=" * 90)

feature_columns = [
    "Price_Lag_1M",
    "Price_Lag_2M",
    "Price_Lag_3M",
    "Price_Lag_6M",
    "Price_Lag_12M",
    "Rolling_Mean_3M",
    "Rolling_Mean_6M",
    "Rolling_Mean_12M",
    "Rolling_Median_3M",
    "Rolling_Median_6M",
    "Rolling_Median_12M",
    "Rolling_Std_3M",
    "Rolling_Std_6M",
    "Rolling_Std_12M",
    "Rolling_Min_6M",
    "Rolling_Max_6M",
    "Historical_Mean_Price",
    "Historical_Median_Price",
    "Historical_Std_Price",
    "Historical_Month_Mean",
    "Historical_Month_Median",
]

feature_missing = (
    forecast_df[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nMissing values in historical/time-series features:")
print(feature_missing.to_string())


# ============================================================
# 26. CHECK FOR INFINITE VALUES
# ============================================================

numeric_columns = forecast_df.select_dtypes(
    include=[np.number]
).columns

infinite_counts = np.isinf(
    forecast_df[numeric_columns]
).sum()

infinite_counts = (
    infinite_counts[infinite_counts > 0]
    .sort_values(ascending=False)
)

print("\nInfinite values:")
if len(infinite_counts) == 0:
    print("None")
else:
    print(infinite_counts.to_string())


# ============================================================
# 27. FEATURE LIST
# ============================================================

target_columns = [
    "Target_Next_Month_Price",
    "Target_Next_Month_Log_Price",
    "Target_Next_Month_Change",
]

excluded_columns = [
    "Target_Date",
]

identifier_columns = [
    "Crop",
    "Month_Start",
    "Year",
    "Month",
    "Quarter",
    "Season",
]

model_feature_columns = [
    col
    for col in forecast_df.columns
    if col not in target_columns
    and col not in excluded_columns
    and col not in [
        "Median_Price",
        "Mean_Price",
        "Min_Price",
        "Max_Price",
    ]
]

print("\n" + "=" * 90)
print("FEATURE SUMMARY")
print("=" * 90)

print(
    f"Total columns: {len(forecast_df.columns)}"
)

print(
    f"Potential model features: "
    f"{len(model_feature_columns)}"
)

print("\nModel feature columns:")
for i, col in enumerate(model_feature_columns, 1):
    print(f"{i:02d}. {col}")


# ============================================================
# 28. CROP-WISE FORECASTING DATASET SUMMARY
# ============================================================

crop_feature_summary = (
    forecast_df.groupby("Crop")
    .agg(
        Rows=("Crop", "size"),
        First_Date=("Month_Start", "min"),
        Last_Date=("Month_Start", "max"),
        Target_Available=("Target_Next_Month_Price", "count"),
        Median_Target=("Target_Next_Month_Price", "median"),
        Mean_Target=("Target_Next_Month_Price", "mean"),
    )
    .sort_values("Rows", ascending=False)
)

print("\n" + "=" * 90)
print("FORECASTING DATASET BY CROP")
print("=" * 90)

print(
    crop_feature_summary.round(2).to_string()
)


# ============================================================
# 29. TIME-ORDER VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("TIME-ORDER VALIDATION")
print("=" * 90)

time_errors = 0

for crop, crop_df in forecast_df.groupby("Crop"):
    crop_df = crop_df.sort_values("Month_Start")

    if not crop_df["Month_Start"].is_monotonic_increasing:
        time_errors += 1

print(
    f"Crops with invalid chronological ordering: "
    f"{time_errors}"
)

# Check target date relationship
target_date_errors = (
    forecast_df["Target_Date"]
    != (
        forecast_df["Month_Start"]
        + pd.DateOffset(months=1)
    )
).sum()

print(
    f"Rows with invalid target-date relationship: "
    f"{target_date_errors:,}"
)


# ============================================================
# 30. SAVE FORECASTING DATASET
# ============================================================

forecast_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 90)
print("FORECASTING FEATURE DATASET SAVED")
print("=" * 90)

print(f"Path: {OUTPUT_FILE}")
print(f"Rows: {len(forecast_df):,}")
print(f"Columns: {len(forecast_df.columns)}")
print(
    f"File size: "
    f"{OUTPUT_FILE.stat().st_size / (1024 ** 2):.2f} MB"
)


# ============================================================
# 31. SAVE FEATURE LIST
# ============================================================

feature_list_file = (
    OUTPUT_DIR
    / "forecasting_feature_columns.csv"
)

pd.DataFrame({
    "Feature": model_feature_columns
}).to_csv(
    feature_list_file,
    index=False
)

print(
    f"\nFeature list saved to:\n"
    f"{feature_list_file}"
)


# ============================================================
# 32. FINAL DATASET PREVIEW
# ============================================================

print("\n" + "=" * 90)
print("FINAL FORECASTING DATASET PREVIEW")
print("=" * 90)

preview_columns = [
    "Crop",
    "Month_Start",
    "Median_Price",
    "Price_Lag_1M",
    "Price_Lag_3M",
    "Price_Lag_12M",
    "Rolling_Mean_3M",
    "Rolling_Mean_6M",
    "Rolling_Std_6M",
    "Month_Sin",
    "Month_Cos",
    "Target_Next_Month_Price",
]

print(
    forecast_df[preview_columns]
    .head(15)
    .round(2)
    .to_string(index=False)
)


# ============================================================
# 33. FINAL VALIDATION SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 90)

print(f"Original cleaned observations: {len(df):,}")
print(f"Monthly crop observations: {len(monthly):,}")
print(f"Forecast-ready observations: {len(forecast_df):,}")
print(f"Number of crops: {forecast_df['Crop'].nunique()}")
print(f"Number of model features: {len(model_feature_columns)}")
print(f"Target: Next month's median crop price")

print("\nLeakage protections:")
print("✓ Lag features use historical observations only")
print("✓ Rolling features are shifted by one month")
print("✓ Historical expanding statistics exclude current target")
print("✓ Seasonal historical statistics exclude current observation")
print("✓ Target is the following month's price")
print("✓ Final month of each crop is excluded from supervised training")

print("\nData treatment:")
print("✓ Extreme prices retained")
print("✓ Extreme prices explicitly flagged")
print("✓ Log price feature created")
print("✓ Potato coverage cutoff preserved")
print("✓ No missing Potato prices were invented")
print("✓ Original cleaned dataset was not modified")

print("\n" + "=" * 90)
print("READY FOR MODEL DEVELOPMENT")
print("=" * 90)